In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset

W0314 11:41:28.630000 5900 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
data = [
    {'text': 'Patient: I have a headache.\nDoctor: Please take rest and stay hydrated.'},
    {'text': 'Patient: My stomach is upset.\nDoctor: Try a light diet and monitor your symptoms.'},
    {'text': 'Patient: I feel dizzy.\nDoctor: Make sure you sit down and drink some water.'}
]
print("Data list restored successfully.")

Data list restored successfully.


In [3]:
dataset = Dataset.from_list(data)

model_name = "deepseek-ai/deepseek-coder-1.3b-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Check if bfloat16 is supported, otherwise use float16
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16)
else:
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16)
print(f"Model and tokenizer initialized with {model_name}")

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Model and tokenizer initialized with deepseek-ai/deepseek-coder-1.3b-instruct


In [4]:
from peft import LoraConfig, get_peft_model

# Define the LoRA Configuration
lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], # Specific layers to attach adapters to
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap your base model with the LoRA config
model = get_peft_model(model, lora_config)

# Verify that only a small portion of the model is now "trainable"
model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 1,348,044,800 || trainable%: 0.1167


In [5]:
training_args = TrainingArguments(
    output_dir="./output_folder",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    max_steps=10,
    logging_steps=1,
   # no_cuda=True,
    fp16=False
)
print("TrainingArguments output_dir updated to './output_folder'.")

TrainingArguments output_dir updated to './output_folder'.


In [6]:
model.save_pretrained('./output_folder')
print("LoRA adapter save path updated to './output_folder'.")

LoRA adapter save path updated to './output_folder'.


In [7]:
model.save_pretrained('./output_folder')
print("END-OF-PROGRAM")

END-OF-PROGRAM


# test the generated Lora adapters in seperate program